In [2]:
import hydra
import numpy as np
import pandas as pd
from pathlib import Path
import anndata as ad
from pathlib import Path
from omegaconf import DictConfig

# Path to the pkl file with new embeddings
PKL_PATH = Path("/home/matthew-mella/valinor/foundation-models-perturbation/molecule_embeddings/tahoe_sci_op3_updated.pkl")


def get_valid_cids():
    """Load pkl and return set of valid pubchem_cids (compounds with LPM_emb)."""
    pkl_df = pd.read_pickle(PKL_PATH)
    tahoe_pkl = pkl_df[pkl_df["dataset"] == "tahoe"]
    tahoe_pkl_lpm = tahoe_pkl[tahoe_pkl["LPM_emb"].notna()].drop_duplicates(
        subset="pubchem_cid", keep="first"
    )
    return set(tahoe_pkl_lpm["pubchem_cid"].astype(int).values)


def load_pkl_embeddings(emb_name):
    """Load embedding lookup from pkl. Returns dict: pubchem_cid -> np.array."""
    pkl_df = pd.read_pickle(PKL_PATH)
    tahoe_pkl = pkl_df[pkl_df["dataset"] == "tahoe"]
    tahoe_pkl_lpm = tahoe_pkl[tahoe_pkl["LPM_emb"].notna()].drop_duplicates(
        subset="pubchem_cid", keep="first"
    )
    lookup = {}
    for _, row in tahoe_pkl_lpm.iterrows():
        cid = int(row["pubchem_cid"])
        lookup[cid] = np.asarray(row[emb_name], dtype=np.float64)
    return lookup


def filter_to_valid(adata, valid_cids):
    """Filter an AnnData object to only compounds in valid_cids."""
    mask = adata.obs["pert_id"].astype(int).isin(valid_cids)
    return adata[mask].copy()

In [1]:
from pathlib import Path
import pandas as pd

import hydra
import torch
import numpy as np
import anndata as ad
from omegaconf import DictConfig


from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler

from benchmark.benchmark import logistic_regression
from benchmark import BenchmarkTask
from benchmark import paths

# Path to the pkl file with new embeddings
PKL_PATH = Path(__file__).resolve().parent.parent.parent.parent / "molecule_embeddings" / "tahoe_sci_op3_updated.pkl"


def get_valid_cids():
    """Load pkl and return set of valid pubchem_cids (compounds with LPM_emb)."""
    pkl_df = pd.read_pickle(PKL_PATH)
    tahoe_pkl = pkl_df[pkl_df["dataset"] == "tahoe"]
    tahoe_pkl_lpm = tahoe_pkl[tahoe_pkl["LPM_emb"].notna()].drop_duplicates(
        subset="pubchem_cid", keep="first"
    )
    return set(tahoe_pkl_lpm["pubchem_cid"].astype(int).values)


def load_pkl_embeddings(emb_name):
    """Load embedding lookup from pkl. Returns dict: pubchem_cid -> np.array."""
    pkl_df = pd.read_pickle(PKL_PATH)
    tahoe_pkl = pkl_df[pkl_df["dataset"] == "tahoe"]
    tahoe_pkl_lpm = tahoe_pkl[tahoe_pkl["LPM_emb"].notna()].drop_duplicates(
        subset="pubchem_cid", keep="first"
    )
    lookup = {}
    for _, row in tahoe_pkl_lpm.iterrows():
        cid = int(row["pubchem_cid"])
        lookup[cid] = np.asarray(row[emb_name], dtype=np.float64)
    return lookup


def filter_to_valid(adata, valid_cids):
    """Filter an AnnData object to only compounds in valid_cids."""
    mask = adata.obs["pert_id"].astype(int).isin(valid_cids)
    return adata[mask].copy()

@hydra.main(version_base=None, config_path="config/tahoe", config_name="config_tahoe_deg")
def main(cfg: DictConfig):

    # Get the set of valid compounds (those with LPM embeddings in pkl)
    valid_cids = get_valid_cids()

    # Load the training and test data
    task = BenchmarkTask(cfg.task_name, f"{cfg.cell_line}.{cfg.fold_id}")
    train, test = task.setup()

    # Restrict to valid compounds
    train = filter_to_valid(train, valid_cids)
    test = filter_to_valid(test, valid_cids)

    # Update task.test so submit() evaluates on restricted set
    task.test = test

    # Load embeddings
    emb_name = cfg.emb_name
    if emb_name == "pca":
        pca_emb = PCA(n_components=100).fit_transform(np.concatenate((train.X, test.X)))
        train_emb = pca_emb[: train.shape[0]]
        test_emb = pca_emb[train.shape[0]:]
    elif emb_name == "random":
        train_emb = np.random.random((train.X.shape[0], 100))
        test_emb = np.random.random((test.X.shape[0], 100))
    elif cfg.emb_name in ("ECFP:2_pkl", "LPM_emb"):
        # Load from pkl file
        pkl_col = "ECFP:2" if cfg.emb_name == "ECFP:2_pkl" else "LPM_emb"
        emb_name = cfg.emb_name
        emb_lookup = load_pkl_embeddings(pkl_col)
        train_emb = np.stack([emb_lookup[int(cid)] for cid in train.obs["pert_id"].astype(int)])
        test_emb = np.stack([emb_lookup[int(cid)] for cid in test.obs["pert_id"].astype(int)])
    else:
        compound_embeddings = ad.read_h5ad(paths.TAHOE_DRUG_EMBEDDINGS)
        compound_embeddings.obs["pubchem_cid"] = compound_embeddings.obs_names.astype(np.int64)
        train_emb = compound_embeddings[train.obs["pert_id"].astype(int).astype(str).tolist()].obsm[emb_name]
        test_emb = compound_embeddings[test.obs["pert_id"].astype(int).astype(str).tolist()].obsm[emb_name]

    assert train_emb.shape[0] == train.n_obs
    assert test_emb.shape[0] == test.n_obs

    # Define estimator pipeline
    assert cfg.estimator_name in ["logistic_regression", "no_change", "prior", "most_frequent"]
    if cfg.estimator_name == "no_change":
        estimator = DummyClassifier(strategy="constant", constant=np.ones(test.n_vars))
    elif cfg.estimator_name == "prior":
        estimator = DummyClassifier(strategy="prior")
    elif cfg.estimator_name == "most_frequent":
        estimator = DummyClassifier(strategy="most_frequent")
    elif cfg.estimator_name == "logistic_regression":
        model = Pipeline([
            ("scale", StandardScaler()),
            ("pca", PCA(n_components=100)),
            ("classifier", logistic_regression.LogisticRegression(C=cfg.model.C, balance_loss=cfg.model.balance_loss)),
        ])
        no_scaling_model = Pipeline([
            ("classifier", logistic_regression.LogisticRegression(C=cfg.model.C, balance_loss=cfg.model.balance_loss)),
        ])
        estimator = no_scaling_model if emb_name == "pca" else model
    
    if cfg.estimator_name in {"prior", "most_frequent", "no_change"}:
        # Pad the prediction with most frequent,
        # and add one example of each class so predictions always have 3 classes
        dummy_train_examples = train.X + 1
        dummy_train_examples = np.concatenate((
            dummy_train_examples,
            np.repeat(np.arange(3)[:, None], dummy_train_examples.shape[1], axis=1),
        ))
        backup_train_emb = np.random.random((dummy_train_examples.shape[0], 100))
        backup_test_emb = np.random.random((test.X.shape[0], 100))

        # there is one sample of each class added on to make sure the dummy classifiers
        # predicts the right number of classes. This doesn't effect most_frequent and 
        # no_change but does slightly regularize prior
        estimator.fit(backup_train_emb, dummy_train_examples)
        predictions = torch.Tensor(np.stack(estimator.predict_proba(backup_test_emb), axis=1)).transpose(2, 1)
    else:
        estimator.fit(train_emb, train.X + 1)
        predictions = estimator.predict_proba(test_emb).transpose(2, 1)

    # Name the model
    if cfg.estimator_name in {"prior", "most_frequent", "no_change"}:
        name = cfg.estimator_name + " baseline"
    else:
        name = cfg.estimator_name + "_baseline_" + emb_name

    description = f"embedding: {emb_name}\nmodel: {estimator}\npca_components=100"

    # Evaluate the predictions
    return task.submit(predictions, name=f"{cfg.estimator_name}_{cfg.emb_name}", description=description)


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'benchmark'

In [6]:
def load_pkl_embeddings(emb_name):
    """Load embedding lookup from pkl. Returns dict: pubchem_cid -> np.array."""
    pkl_df = pd.read_pickle(PKL_PATH)
    tahoe_pkl = pkl_df[pkl_df["dataset"] == "tahoe"]
    tahoe_pkl_lpm = tahoe_pkl[tahoe_pkl["LPM_emb"].notna()].drop_duplicates(
        subset="pubchem_cid", keep="first"
    )
    lookup = {}
    for _, row in tahoe_pkl_lpm.iterrows():
        cid = int(row["pubchem_cid"])
        lookup[cid] = np.asarray(row[emb_name], dtype=np.float64)
    return lookup

lookup = load_pkl_embeddings("LPM_emb")

In [9]:
lookup

{56649450: array([-0.06710507,  0.01151098, -0.27214149, -0.07315319, -0.02768727,
        -0.17768914,  0.39222875,  0.14058456, -0.16213872,  0.09113336,
         0.12844981,  0.22089913, -0.01264779,  0.05188127, -0.08423468,
         0.03291929,  0.05088047,  0.09035806,  0.15678512,  0.09947049,
        -0.02055788,  0.00268137, -0.23401576, -0.29663792,  0.01079649,
         0.13336736, -0.11460651, -0.36539119, -0.21403676, -0.04252561,
         0.10939903,  0.02326885,  0.05544936, -0.05371497, -0.12366473,
        -0.16091105, -0.15626857,  0.06683911,  0.25579849, -0.05936127,
         0.01854019, -0.19237383, -0.29371276,  0.05730662,  0.06753719,
         0.00262417, -0.15479104, -0.11198191, -0.29056546,  0.05753882,
         0.04888156,  0.07959833,  0.20711212, -0.04820584,  0.08460071,
         0.06295132,  0.19035704,  0.0693324 , -0.08787198,  0.20672281,
        -0.0225003 , -0.23767431,  0.13321386,  0.04694653, -0.23070379,
         0.34392858, -0.1528938 , -0.1993